Question → FAISS Top-20 + BM25 Top-20 → Candidate Pool → Cross-Encoder → Score Fusion → Top-5


In [1]:
# Install dependencies
!pip -q install sentence-transformers faiss-cpu rank-bm25 pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import re
import numpy as np
import pandas as pd
import faiss

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display

print("Libraries loaded.")

c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded.


In [3]:
# Paths
chunks_path = "./02_rmit_assessment_support_chunks.csv"
test_path = "./test_prompts.json"

# Load chunks
chunks_df = pd.read_csv(chunks_path)

# Load test prompts
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_df = pd.DataFrame(test_data["tests"])

print("Chunks:", len(chunks_df))
print("Test prompts:", len(test_df))


Chunks: 71
Test prompts: 12


## 1. Dense retrieval

Use the same embedding model and document representation as Iteration 2/3, but retrieve 20 candidates so that the reranking stage has a larger candidate pool.


In [4]:
# Structured embedding text
chunks_df["embedding_text"] = (
    "Title: " + chunks_df["title"].fillna("") + ". " +
    "Category: " + chunks_df["category"].fillna("") + ". " +
    "Section: " + chunks_df["section"].fillna("") + ". " +
    "Content: " + chunks_df["content"].fillna("")
)

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(embeddings)

faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
faiss_index.add(embeddings)

print("FAISS vectors:", faiss_index.ntotal)


Batches: 100%|██████████| 3/3 [00:00<00:00,  3.03it/s]

FAISS vectors: 71


## 2. BM25 retrieval


In [5]:
def tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

bm25_corpus = [
    tokenize(text)
    for text in chunks_df["embedding_text"]
]

bm25 = BM25Okapi(bm25_corpus)

print("BM25 documents:", len(bm25_corpus))


BM25 documents: 71


## 3. Cross-encoder reranker


In [6]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

print("Reranker loaded.")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5397.08it/s]


Reranker loaded.


In [7]:
def dense_candidates(question, k=20):
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(
        query_embedding,
        k
    )

    return [
        {
            "index": int(idx),
            "dense_score": float(score),
            "dense_rank": rank + 1
        }
        for rank, (idx, score) in enumerate(zip(indices[0], scores[0]))
    ]


def bm25_candidates(question, k=20):
    query_tokens = tokenize(question)
    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:k]

    return [
        {
            "index": int(idx),
            "bm25_score": float(scores[idx]),
            "bm25_rank": rank + 1
        }
        for rank, idx in enumerate(top_indices)
    ]


## 4. Score fusion

The key difference from Iteration 3 is that the final ranking does not rely only on the cross-encoder.
We combine:

- Dense retrieval signal
- BM25 signal
- Cross-encoder relevance signal
  Each score is min-max normalised within the candidate pool.
  Default weights:
- Dense = 0.30
- BM25 = 0.20
- Reranker = 0.50
  The reranker remains the strongest signal, but the original retrieval evidence is retained.


In [8]:
def minmax_normalize(values):
    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return values

    minimum = values.min()
    maximum = values.max()

    if maximum - minimum < 1e-12:
        return np.ones(len(values))

    return (values - minimum) / (maximum - minimum)


def hybrid_score_fusion(
    question,
    dense_k=20,
    bm25_k=20,
    final_k=5,
    dense_weight=0.30,
    bm25_weight=0.20,
    reranker_weight=0.50
):
    dense_results = dense_candidates(question, dense_k)
    bm25_results = bm25_candidates(question, bm25_k)

    # Candidate union
    candidate_indices = sorted(
        set(
            [x["index"] for x in dense_results] +
            [x["index"] for x in bm25_results]
        )
    )

    dense_lookup = {
        x["index"]: x for x in dense_results
    }

    bm25_lookup = {
        x["index"]: x for x in bm25_results
    }

    # Values for all candidates
    dense_values = [
        dense_lookup.get(idx, {}).get("dense_score", 0.0)
        for idx in candidate_indices
    ]

    bm25_values = [
        bm25_lookup.get(idx, {}).get("bm25_score", 0.0)
        for idx in candidate_indices
    ]

    # Cross-encoder
    pairs = [
        (
            question,
            chunks_df.iloc[idx]["embedding_text"]
        )
        for idx in candidate_indices
    ]

    reranker_values = reranker.predict(pairs)

    # Normalise all signals
    dense_norm = minmax_normalize(dense_values)
    bm25_norm = minmax_normalize(bm25_values)
    reranker_norm = minmax_normalize(reranker_values)

    # Weighted fusion
    fused_scores = (
        dense_weight * dense_norm +
        bm25_weight * bm25_norm +
        reranker_weight * reranker_norm
    )

    ranked_positions = np.argsort(fused_scores)[::-1][:final_k]

    results = []

    for position in ranked_positions:
        idx = candidate_indices[position]
        row = chunks_df.iloc[idx]

        results.append({
            "index": idx,
            "chunk_id": row["chunk_id"],
            "source_id": row["source_id"],
            "title": row["title"],
            "category": row["category"],
            "section": row["section"],
            "content": row["content"],
            "dense_score": dense_values[position],
            "bm25_score": bm25_values[position],
            "reranker_score": float(reranker_values[position]),
            "dense_norm": dense_norm[position],
            "bm25_norm": bm25_norm[position],
            "reranker_norm": reranker_norm[position],
            "fused_score": float(fused_scores[position])
        })

    return pd.DataFrame(results)


## 5. Inspect the three previously problematic questions

These are the cases where Iteration 3 failed to retrieve the expected chunks in Top-5.


In [9]:
problem_questions = [
    (
        "Bereavement",
        "My nan just passed away but I've got a group assignment due friday, can I even apply for an extension on something like this? How much extra time could I get?",
        ["POL-01_62", "EXT-01_1"]
    ),
    (
        "Plagiarism accusation",
        "I think my tutor marked me down assuming I plagiarised, very upset about this because I definitely didn't, how should I go about correcting this issue?",
        ["POL-01_58", "POL-01_59"]
    ),
    (
        "ELP extension",
        "The extensions adjustment is on my ELP, but the assignment brief also says to fill out the standard extension form, do I need to do both, or is my learning plan enough on its own",
        ["EXT-01_2", "EAA-02_39"]
    )
]

for name, question, expected in problem_questions:
    print("\n" + "=" * 80)
    print(name)
    print("Expected:", expected)

    results = hybrid_score_fusion(question)

    display(
        results[
            [
                "chunk_id",
                "dense_norm",
                "bm25_norm",
                "reranker_norm",
                "fused_score"
            ]
        ]
    )



Bereavement
Expected: ['POL-01_62', 'EXT-01_1']


,chunk_id,dense_norm,bm25_norm,reranker_norm,fused_score
0,EAA-02_41,0.880324,0.657651,1.000000,0.895627
1,EAA-02_34,0.904967,1.000000,0.788764,0.865872
2,EXT-01_0,1.000000,0.497256,0.753832,0.776367
3,EAA-02_35,0.869426,0.501999,0.472654,0.597554
4,SC-01_6,0.591187,0.511629,0.532049,0.545706



Plagiarism accusation
Expected: ['POL-01_58', 'POL-01_59']


,chunk_id,dense_norm,bm25_norm,reranker_norm,fused_score
0,SC-01_16,0.758363,0.870875,0.909699,0.856533
1,POL-01_70,0.959486,0.000000,1.000000,0.787846
2,EAA-02_51,0.603900,0.944317,0.210666,0.475366
3,EAA-02_27,0.678271,1.000000,0.138468,0.472715
4,EAA-02_53,0.757604,0.931206,0.056404,0.441725



ELP extension
Expected: ['EXT-01_2', 'EAA-02_39']


,chunk_id,dense_norm,bm25_norm,reranker_norm,fused_score
0,EAA-02_34,1.000000,0.840375,1.000000,0.968075
1,EAA-02_39,0.915747,1.000000,0.700085,0.824767
2,EAA-02_36,0.989514,0.492199,0.850161,0.820375
3,EAA-02_40,0.893133,0.629192,0.644140,0.715848
4,EAA-02_41,0.881393,0.000000,0.778001,0.653419


## 6. Evaluation


In [10]:
def any_expected_retrieved(retrieved, expected):
    return any(chunk in retrieved for chunk in expected)


def all_expected_retrieved(retrieved, expected):
    return all(chunk in retrieved for chunk in expected)


def evaluate_iteration4(test_df):
    results = []

    for _, test in test_df.iterrows():

        question = test["prompt"]
        expected_chunks = test["chunks"]

        retrieved_df = hybrid_score_fusion(
            question,
            dense_k=20,
            bm25_k=20,
            final_k=5
        )

        retrieved_chunks = retrieved_df["chunk_id"].tolist()

        top1 = retrieved_chunks[:1]
        top3 = retrieved_chunks[:3]
        top5 = retrieved_chunks[:5]

        expected_ranks = [
            retrieved_chunks.index(chunk) + 1
            for chunk in expected_chunks
            if chunk in retrieved_chunks
        ]

        results.append({
            "prompt": question,
            "expected_chunks": expected_chunks,

            "top1_chunk": top1[0] if top1 else None,
            "top3_chunks": top3,
            "top5_chunks": top5,

            "expected_rank_in_top5":
                min(expected_ranks) if expected_ranks else None,

            "top1_hit":
                any_expected_retrieved(top1, expected_chunks),
            "top3_hit":
                any_expected_retrieved(top3, expected_chunks),
            "top5_hit":
                any_expected_retrieved(top5, expected_chunks),

            "top1_all_hit":
                all_expected_retrieved(top1, expected_chunks),
            "top3_all_hit":
                all_expected_retrieved(top3, expected_chunks),
            "top5_all_hit":
                all_expected_retrieved(top5, expected_chunks)
        })

    return pd.DataFrame(results)


In [11]:
iteration4_df = evaluate_iteration4(test_df)

print("ITERATION 4 — HYBRID + SCORE FUSION")
print("------------------------------------")

print("\nAny expected chunk retrieved:")
print(f"Top-1: {iteration4_df['top1_hit'].mean():.1%}")
print(f"Top-3: {iteration4_df['top3_hit'].mean():.1%}")
print(f"Top-5: {iteration4_df['top5_hit'].mean():.1%}")

print("\nAll expected chunks retrieved:")
print(f"Top-1: {iteration4_df['top1_all_hit'].mean():.1%}")
print(f"Top-3: {iteration4_df['top3_all_hit'].mean():.1%}")
print(f"Top-5: {iteration4_df['top5_all_hit'].mean():.1%}")


ITERATION 4 — HYBRID + SCORE FUSION
------------------------------------

Any expected chunk retrieved:
Top-1: 58.3%
Top-3: 83.3%
Top-5: 83.3%

All expected chunks retrieved:
Top-1: 50.0%
Top-3: 75.0%
Top-5: 75.0%


In [12]:
# Compare all iterations
comparison_df = pd.DataFrame({
    "Metric": [
        "Any expected — Top-1",
        "Any expected — Top-3",
        "Any expected — Top-5",
        "All expected — Top-1",
        "All expected — Top-3",
        "All expected — Top-5"
    ],
    "Iteration 2": [
        0.500, 0.750, 0.917,
        0.417, 0.583, 0.750
    ],
    "Iteration 3": [
        0.583, 0.750, 0.833,
        0.500, 0.750, 0.750
    ],
    "Iteration 4": [
        iteration4_df["top1_hit"].mean(),
        iteration4_df["top3_hit"].mean(),
        iteration4_df["top5_hit"].mean(),
        iteration4_df["top1_all_hit"].mean(),
        iteration4_df["top3_all_hit"].mean(),
        iteration4_df["top5_all_hit"].mean()
    ]
})

comparison_df["I4 vs I2"] = (
    comparison_df["Iteration 4"] -
    comparison_df["Iteration 2"]
)

comparison_df["I4 vs I3"] = (
    comparison_df["Iteration 4"] -
    comparison_df["Iteration 3"]
)

display(
    comparison_df.style.format({
        "Iteration 2": "{:.1%}",
        "Iteration 3": "{:.1%}",
        "Iteration 4": "{:.1%}",
        "I4 vs I2": "{:+.1%}",
        "I4 vs I3": "{:+.1%}"
    })
)


,Metric,Iteration 2,Iteration 3,Iteration 4,I4 vs I2,I4 vs I3
0,Any expected — Top-1,50.0%,58.3%,58.3%,+8.3%,+0.0%
1,Any expected — Top-3,75.0%,75.0%,83.3%,+8.3%,+8.3%
2,Any expected — Top-5,91.7%,83.3%,83.3%,-8.4%,+0.0%
3,All expected — Top-1,41.7%,50.0%,50.0%,+8.3%,+0.0%
4,All expected — Top-3,58.3%,75.0%,75.0%,+16.7%,+0.0%
5,All expected — Top-5,75.0%,75.0%,75.0%,+0.0%,+0.0%


In [13]:
# Inspect remaining Top-5 failures
top5_failures_4 = iteration4_df[
    iteration4_df["top5_hit"] == False
][[
    "prompt",
    "expected_chunks",
    "top1_chunk",
    "top3_chunks",
    "top5_chunks",
    "expected_rank_in_top5"
]]

print("Iteration 4 Top-5 failures:", len(top5_failures_4))
display(top5_failures_4)


Iteration 4 Top-5 failures: 2


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EAA-02_41,"[EAA-02_41, EAA-02_34, EXT-01_0]","[EAA-02_41, EAA-02_34, EXT-01_0, EAA-02_35, SC...",NaN
10,I think my tutor marked me down assuming I pla...,"[POL-01_58, POL-01_59]",SC-01_16,"[SC-01_16, POL-01_70, EAA-02_51]","[SC-01_16, POL-01_70, EAA-02_51, EAA-02_27, EA...",NaN


In [15]:
# Save results
iteration4_df.to_csv(
    "./07_iteration4_score_fusion_results.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_df.to_csv(
    "./07_iteration2_vs_iteration4_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Results saved.")


Results saved.


In [16]:
def inspect_candidate_ranks(question, expected_chunks, k=20):

    dense_results = dense_candidates(question, k)
    bm25_results = bm25_candidates(question, k)

    dense_lookup = {
        chunks_df.iloc[x["index"]]["chunk_id"]: x["dense_rank"]
        for x in dense_results
    }

    bm25_lookup = {
        chunks_df.iloc[x["index"]]["chunk_id"]: x["bm25_rank"]
        for x in bm25_results
    }

    rows = []

    for chunk in expected_chunks:
        rows.append({
            "expected_chunk": chunk,
            "faiss_rank": dense_lookup.get(chunk),
            "bm25_rank": bm25_lookup.get(chunk)
        })

    return pd.DataFrame(rows)

In [17]:
q1 = "My nan just passed away but I've got a group assignment due friday, can I even apply for an extension on something like this? How much extra time could I get?"

display(
    inspect_candidate_ranks(
        q1,
        ["POL-01_62", "EXT-01_1"]
    )
)

,expected_chunk,faiss_rank,bm25_rank
0,POL-01_62,6,18
1,EXT-01_1,2,19


In [18]:
q2 = "I think my tutor marked me down assuming I plagiarised, very upset about this because I definitely didn't, how should I go about correcting this issue?"

display(
    inspect_candidate_ranks(
        q2,
        ["POL-01_58", "POL-01_59"]
    )
)

,expected_chunk,faiss_rank,bm25_rank
0,POL-01_58,11,None
1,POL-01_59,3,None
